## Observability with langsmith

### Set API Keys

In [3]:
import os
from dotenv import load_dotenv
load_dotenv()

True

In [4]:
print("LANGSMITH_TRACING :", os.getenv("LANGSMITH_TRACING", "not set"))
print("LANGSMITH_API_KEY :", "✅" if os.getenv("LANGSMITH_API_KEY") else "❌  missing")
print("LANGSMITH_PROJECT :", os.getenv("LANGSMITH_PROJECT", "default"))
print("GROQ_API_KEY      :", "✅" if os.getenv("GROQ_API_KEY")      else "❌  missing")
print("OPENAI_API_KEY    :", "✅" if os.getenv("OPENAI_API_KEY")    else "❌  missing")

LANGSMITH_TRACING : true
LANGSMITH_API_KEY : ✅
LANGSMITH_PROJECT : abc-project
GROQ_API_KEY      : ✅
OPENAI_API_KEY    : ✅


### Experiment 1 — First Auto-Traced LangChain Call

Set the three environment variables and every LangChain call is automatically sent to
LangSmith. No code changes whatsoever.

```
LANGSMITH_TRACING=true         ← master switch
LANGSMITH_API_KEY=...          ← your LangSmith API key
LANGSMITH_PROJECT=my-project   ← groups related traces (optional)
```

After running this cell: go to smith.langchain.com → your project → you'll see the run.

In [5]:
from langchain_groq import ChatGroq

llm = ChatGroq(model="openai/gpt-oss-120b", temperature=0.3)

response = llm.invoke("What is a LangSmith Run? Answer in 2 sentences.")
print(response.content)

A LangSmith Run is an execution trace generated by LangSmith, the observability platform for LLM applications, which records the inputs, outputs, prompts, and metadata of a single interaction with a language model. It enables developers to monitor, debug, and analyze the performance and behavior of their LLM-powered workflows.


### Experiment 2 — @traceable: Make Your Own Functions Visible in LangSmith

LangSmith auto-traces LangChain objects. For your own Python functions, use the `@traceable` decorator.

```
Without @traceable:   LangSmith sees only the LLM call — no context around it

With @traceable:      LangSmith sees your full function as a parent Run,
                      with the LLM call nested inside as a child Run
```

**`run_type` values:**
| Value | When to use |
|-------|------------|
| `"llm"` | Function that calls a language model |
| `"tool"` | Function that retrieves data, searches, or calls an API |
| `"chain"` | Orchestrator function that calls other functions |

In [9]:
import re
from langsmith import traceable


# ── Tool: keyword search over the real document ────────────────────────────
@traceable(run_type="tool", name="doc_keyword_search")
def search_document(query: str, top_k: int = 3) -> list:
    """Searches llm_production_guide.txt by keyword overlap. Visible as a Tool Run."""
    with open("../data/llm_production_guide.txt", encoding="utf-8") as f:
        text = f.read()
    paragraphs = [p.strip() for p in text.split("\n\n") if len(p.strip()) > 80]
    keywords   = set(re.findall(r"\b\w{4,}\b", query.lower()))
    ranked     = sorted(paragraphs,
                        key=lambda p: sum(1 for kw in keywords if kw in p.lower()),
                        reverse=True)
    return ranked[:top_k]

In [10]:
# ── Chain: orchestrates search → LLM → answer ─────────────────────────────
@traceable(run_type="chain", name="doc_qa_pipeline")
def doc_qa(question: str) -> str:
    """Parent chain. LangSmith shows: doc_qa_pipeline → doc_keyword_search + ChatGroq."""
    sections = search_document(question)            # ← child Tool Run
    context  = "\n\n".join(sections)
    prompt   = f"Context:\n{context}\n\nQuestion: {question}\nAnswer concisely:"
    return llm.invoke(prompt).content               # ← child LLM Run

In [11]:
answer = doc_qa("What are the main LLM security threats?")
print(f"Answer: {answer[:300]}...")

Answer: **Main LLM security threats (as highlighted by the OWASP Top 10 for LLM applications)**  

1. **Prompt / Injection attacks** – crafted inputs that cause the model to behave maliciously or reveal hidden data.  
2. **Data leakage / Privacy exposure** – unintended disclosure of training‑set or user‑pro...


### Experiment 3 — Enrich Traces: Tags, Metadata, run_name

Raw traces tell you *what* happened. Tags and metadata tell you *who*, *why*, and *in what context*.

Pass `langsmith_extra=` directly to any `llm.invoke()` or inside a `@traceable` function — no LCEL, no RunnableConfig needed.

| Field | Purpose | Example |
|-------|---------|---------|
| `tags` | String labels — filter in dashboard | `["production", "groq"]` |
| `metadata` | Any key-value dict — visible in run detail | `{"user_id": "alice", "feature": "support-bot"}` |
| `run_name` | Override the default run title | `"support-query-alice"` |

**Real production use:** filter `metadata.user_id = "alice"` to see all of one user's traces and sum their token costs.

In [12]:
from langsmith import get_current_run_tree

from langsmith import get_current_run_tree 

@traceable(run_type="chain", name="support-query")
def support_qa(question: str, user_id: str, session_id: str) -> str:
    run = get_current_run_tree()
    if run:
        run.metadata.update({
            "user_id":    user_id,
            "session_id": session_id,
            "feature":    "customer-support",
            "env":        "production",
        })
        run.tags = ["production", "support-bot", "groq"]
    return llm.invoke(question).content

In [13]:
list_of_queriers = [
    ("priya",   "sess_001", "What is prompt injection and how do we prevent it?"),
    ("aditi",   "sess_002", "What are best practices for LLM output validation?"),
    ("sheetal", "sess_003", "How do we monitor LLM costs in production?"),
]

In [14]:
# Run three different users — each trace is tagged for filtering
for user, session, q in list_of_queriers:
    answer = support_qa(q, user_id=user, session_id=session)
    print(f"[{user}] {answer[:150]}...\n")

[priya] ## Prompt Injection – A Quick Primer

**Prompt injection** (sometimes called a *jailbreak* or *adversarial prompt*) is a class of attacks in which a u...

[aditi] Below is a practical, step‑by‑step guide to **validating the output of Large Language Models (LLMs)**.  
It is organized around the three most common ...

[sheetal] ## Monitoring LLM Costs in Production – A Play‑by‑Play Guide  

Below is a **complete, production‑ready framework** you can copy‑paste into your own o...

